In [1]:
from  datetime import datetime, timedelta
import gc
import numpy as np, pandas as pd
import lightgbm as lgb

#Hiển thị max 50 cột
pd.options.display.max_columns = 50

# Trỏ đường dẫn để gọi file trong thư mục src
import sys, os
sys.path.append(os.path.abspath('../src'))

from dataPreprocessing import create_dt, create_fea

In [2]:
#Định nghĩa dữ liệu cho từng cột phân loại
CAL_DTYPES={"event_name_1": "category", "event_name_2": "category", "event_type_1": "category", 
         "event_type_2": "category", "weekday": "category", 'wm_yr_wk': 'int16', "wday": "int16",
        "month": "int16", "year": "int16", "snap_CA": "float32", 'snap_TX': 'float32', 'snap_WI': 'float32' }

#Định nghĩa dữ liệu cho câc cột lien quan đến giá
PRICE_DTYPES = {"store_id": "category", "item_id": "category", "wm_yr_wk": "int16","sell_price":"float32" }

#Ngày bắt đầu lấy dữ liệu
FIRST_DAY = 350

#Định nghĩa một số biến đầu vào
h = 28 
max_lags = 57
tr_last = 1913
fday = datetime(2016,4, 25) 
fday

datetime.datetime(2016, 4, 25, 0, 0)

In [3]:
#Convert dữ liệu sang dạng long
df = create_dt(is_train=True, first_day= FIRST_DAY, nrows=None, price_dt = PRICE_DTYPES, cal_dt = CAL_DTYPES)
df.shape

(40718219, 22)

In [4]:
#Load phân loại deamand đã có từ trước
df_demandType = pd.read_csv("../dataset/result/demand_classification.csv")
df_demandType.head()

,id,ADI,CV2,Category
0,FOODS_1_001_CA_1_validation,2.29,0.26,Intermittent (Ngắt quãng)
1,FOODS_1_001_CA_2_validation,1.93,0.46,Intermittent (Ngắt quãng)
2,FOODS_1_001_CA_3_validation,2.23,0.50,Lumpy (Cục bộ)
3,FOODS_1_001_CA_4_validation,3.87,0.24,Intermittent (Ngắt quãng)
4,FOODS_1_001_TX_1_validation,3.06,0.29,Intermittent (Ngắt quãng)


In [5]:
#Add demand feature vào df
def add_demandFeature(demand_dir = "../dataset/result/demand_classification.csv", data = None):
    #Load phân loại deamand đã có từ trước
    df_demandType = pd.read_csv(demand_dir)

    # Merge df with df_demandType to add 'Category' column
    df = data

    # Merge df with df_demandType to add 'Category' column
    df = df.merge(df_demandType[['id', 'Category']], on='id', how='left')

    # Perform one-hot encoding on 'Category' column
    df = pd.get_dummies(df, columns=['Category'], prefix='cat')

    return df

df = add_demandFeature(data = df)

df.head()

,id,item_id,dept_id,store_id,cat_id,state_id,d,sales,date,wm_yr_wk,weekday,wday,month,year,event_name_1,event_type_1,event_name_2,event_type_2,snap_CA,snap_TX,snap_WI,sell_price,cat_Erratic (Thất thường),cat_Intermittent (Ngắt quãng),cat_Lumpy (Cục bộ),cat_Smooth (Đều đặn)
0,HOBBIES_1_002_CA_1_validation,1,0,0,0,0,d_350,0.0,2012-01-13,11150,0,7,1,2012,0,0,0,0,0.0,1.0,0.0,3.97,False,True,False,False
1,HOBBIES_1_004_CA_1_validation,3,0,0,0,0,d_350,2.0,2012-01-13,11150,0,7,1,2012,0,0,0,0,0.0,1.0,0.0,4.34,False,True,False,False
2,HOBBIES_1_005_CA_1_validation,4,0,0,0,0,d_350,0.0,2012-01-13,11150,0,7,1,2012,0,0,0,0,0.0,1.0,0.0,2.48,False,True,False,False
3,HOBBIES_1_008_CA_1_validation,7,0,0,0,0,d_350,0.0,2012-01-13,11150,0,7,1,2012,0,0,0,0,0.0,1.0,0.0,0.50,False,False,True,False
4,HOBBIES_1_009_CA_1_validation,8,0,0,0,0,d_350,2.0,2012-01-13,11150,0,7,1,2012,0,0,0,0,0.0,1.0,0.0,1.77,False,True,False,False


In [6]:
#Làm giàu đặc trưng
create_fea(df ,lags_fe = [7, 28], wins_fe = [7, 28])
df.shape

(40718219, 35)

In [7]:
#Peview một số dòng đầu
df.head()

,id,item_id,dept_id,store_id,cat_id,state_id,d,sales,date,wm_yr_wk,weekday,wday,month,year,event_name_1,event_type_1,event_name_2,event_type_2,snap_CA,snap_TX,snap_WI,sell_price,cat_Erratic (Thất thường),cat_Intermittent (Ngắt quãng),cat_Lumpy (Cục bộ),cat_Smooth (Đều đặn),lag_7,lag_28,rmean_7_7,rmean_28_7,rmean_7_28,rmean_28_28,week,quarter,mday
0,HOBBIES_1_002_CA_1_validation,1,0,0,0,0,d_350,0.0,2012-01-13,11150,0,7,1,2012,0,0,0,0,0.0,1.0,0.0,3.97,False,True,False,False,NaN,NaN,NaN,NaN,NaN,NaN,2,1,13
1,HOBBIES_1_004_CA_1_validation,3,0,0,0,0,d_350,2.0,2012-01-13,11150,0,7,1,2012,0,0,0,0,0.0,1.0,0.0,4.34,False,True,False,False,NaN,NaN,NaN,NaN,NaN,NaN,2,1,13
2,HOBBIES_1_005_CA_1_validation,4,0,0,0,0,d_350,0.0,2012-01-13,11150,0,7,1,2012,0,0,0,0,0.0,1.0,0.0,2.48,False,True,False,False,NaN,NaN,NaN,NaN,NaN,NaN,2,1,13
3,HOBBIES_1_008_CA_1_validation,7,0,0,0,0,d_350,0.0,2012-01-13,11150,0,7,1,2012,0,0,0,0,0.0,1.0,0.0,0.50,False,False,True,False,NaN,NaN,NaN,NaN,NaN,NaN,2,1,13
4,HOBBIES_1_009_CA_1_validation,8,0,0,0,0,d_350,2.0,2012-01-13,11150,0,7,1,2012,0,0,0,0,0.0,1.0,0.0,1.77,False,True,False,False,NaN,NaN,NaN,NaN,NaN,NaN,2,1,13


In [8]:
#Drop các dòng không có dữ liệU
df.dropna(inplace = True)
df.shape

(39041269, 35)

In [9]:
#Các cột phân loại
cat_feats = ['item_id', 'dept_id','store_id', 'cat_id', 'state_id'] + \
["event_name_1", "event_name_2", "event_type_1", "event_type_2"] + \
['cat_Erratic (Thất thường)', 'cat_Intermittent (Ngắt quãng)', 'cat_Lumpy (Cục bộ)', 'cat_Smooth (Đều đặn)']
#Các cột không sử dụng
useless_cols = ["id", "date", "sales","d", "wm_yr_wk", "weekday"]


#Định nghĩa param cho thuật toán LightGBM
params = {
        "objective" : "poisson",
        "metric" :"rmse",
        "force_row_wise" : True,
        "learning_rate" : 0.075,
#         "sub_feature" : 0.8,
        "sub_row" : 0.75,
        "bagging_freq" : 1,
        "lambda_l2" : 0.1,
         "nthread" : 8,
        "metric": ["rmse"],
    'verbosity': 1,
    'num_iterations' : 2500,
    'num_leaves': 128,
    "min_data_in_leaf": 100,
    "device": "cpu"
    # "early_stopping_round": 100
}



In [10]:
#Các cột sử dụng trong tập train (ngoại trừ những cột ko dùng ĐN ở trên)
train_cols = df.columns[~df.columns.isin(useless_cols)]

#Tách dữ liệu đặc trưng và mục tiêu trong tập train
X_train = df[train_cols]
y_train = df["sales"]

#Cố định seed random
np.random.seed(777)

#Fake ra 30 000 index cho tập fake valid
fake_valid_inds = np.random.choice(X_train.index.values, 30000, replace = False)

#Index còn lại làm cho tập train
train_inds = np.setdiff1d(X_train.index.values, fake_valid_inds)

#Tập dữ liệu train
train_data = lgb.Dataset(X_train.loc[train_inds] , label = y_train.loc[train_inds], 
                         categorical_feature=cat_feats, free_raw_data=False)

#Tập dữ liệU valid fake
fake_valid_data = lgb.Dataset(X_train.loc[fake_valid_inds], label = y_train.loc[fake_valid_inds],
                              categorical_feature=cat_feats,
                 free_raw_data=False)

#Giải phóng bộ nhớ
del df, X_train, y_train, fake_valid_inds,train_inds ; gc.collect()

0

In [11]:
m_lgb = lgb.train(params, train_data, valid_sets = [fake_valid_data],  callbacks=[lgb.log_evaluation(20)]) 

[LightGBM] [Warning] Categorical features with more bins than the configured maximum bin number found.
[LightGBM] [Warning] For categorical features, max_bin and max_bin_by_feature may be ignored with a large number of categories.
[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] Total Bins 4610
[LightGBM] [Info] Number of data points in the train set: 39011269, number of used features: 29
[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] Start training from score 0.304274
[20]	valid_0's rmse: 3.0297
[40]	valid_0's rmse: 2.70027
[60]	valid_0's rmse: 2.59018
[80]	valid_0's rmse: 2.55196
[100]	valid_0's rmse: 2.53545
[120]	valid_0's rmse: 2.50863
[140]	valid_0's rmse: 2.48702
[160]	valid_0's rmse: 2.45828
[180]	valid_0's rmse: 2.45624
[200]	valid_0's rmse: 2.43058
[220]	valid_0's rmse: 2.4149
[240]	valid_0's rmse: 2.39692
[260]	valid_0's rmse: 2.38403
[280]	valid_0's rmse: 2.37736
[300]	valid_

In [12]:
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

model_path = f"../dataset/result/model_{timestamp}.lgb"

m_lgb.save_model(model_path)

print(f"Saved model to: {model_path}")

Saved model to: ../dataset/result/model_20260327_085207.lgb


In [13]:
#Recursive forecasting và magic multiplier
alphas = [1.028, 1.023, 1.018]
weights = [1/len(alphas)]*len(alphas)
sub = 0.

for icount, (alpha, weight) in enumerate(zip(alphas, weights)):

    #Lấy dữ liệu
    te = create_dt(is_train=False, price_dt = PRICE_DTYPES, cal_dt = CAL_DTYPES, tr_last = tr_last)

    #Add demand feature vào te
    te = add_demandFeature(data = te)
    cols = [f"d_{tr_last + i}" for i in range(1,29)]

    for tdelta in range(0, 28):
        day = fday + timedelta(days=tdelta)
        print(tdelta, day)
        tst = te[(te.date >= day - timedelta(days=max_lags)) & (te.date <= day)].copy()

        create_fea(tst ,lags_fe = [7, 28], wins_fe = [7, 28])
        
        tst = tst.loc[tst.date == day , train_cols]
        te.loc[te.date == day, "sales"] = alpha*m_lgb.predict(tst) # magic multiplier by kyakovlev



    te_sub = te.loc[te.date >= fday, ["id", "sales"]].copy()

    te_sub["F"] = [f"d_{tr_last + rank}" 
               for rank in te_sub.groupby("id")["id"].cumcount() + 1]
    te_sub = te_sub.set_index(["id", "F" ]).unstack()["sales"][cols].reset_index()
    te_sub.fillna(0., inplace = True)
    te_sub.sort_values("id", inplace = True)
    te_sub.reset_index(drop=True, inplace = True)
    te_sub.to_csv(f"../dataset/result/all/submission_all_{icount}.csv",index=False)
    if icount == 0 :
        sub = te_sub
        sub[cols] *= weight
    else:
        sub[cols] += te_sub[cols]*weight
    print(icount, alpha, weight)


sub["id"] = sub["id"].str.extract(r"^(.+?)_(?:validation|evaluation)$", expand=False)
sub.columns.name = None
sub.set_index("id", inplace = True)
sub.to_csv("../dataset/result/all/submission_all.csv",index=False)

0 2016-04-25 00:00:00
1 2016-04-26 00:00:00
2 2016-04-27 00:00:00
3 2016-04-28 00:00:00
4 2016-04-29 00:00:00
5 2016-04-30 00:00:00
6 2016-05-01 00:00:00
7 2016-05-02 00:00:00
8 2016-05-03 00:00:00
9 2016-05-04 00:00:00
10 2016-05-05 00:00:00
11 2016-05-06 00:00:00
12 2016-05-07 00:00:00
13 2016-05-08 00:00:00
14 2016-05-09 00:00:00
15 2016-05-10 00:00:00
16 2016-05-11 00:00:00
17 2016-05-12 00:00:00
18 2016-05-13 00:00:00
19 2016-05-14 00:00:00
20 2016-05-15 00:00:00
21 2016-05-16 00:00:00
22 2016-05-17 00:00:00
23 2016-05-18 00:00:00
24 2016-05-19 00:00:00
25 2016-05-20 00:00:00
26 2016-05-21 00:00:00
27 2016-05-22 00:00:00
0 1.028 0.3333333333333333
0 2016-04-25 00:00:00
1 2016-04-26 00:00:00
2 2016-04-27 00:00:00
3 2016-04-28 00:00:00
4 2016-04-29 00:00:00
5 2016-04-30 00:00:00
6 2016-05-01 00:00:00
7 2016-05-02 00:00:00
8 2016-05-03 00:00:00
9 2016-05-04 00:00:00
10 2016-05-05 00:00:00
11 2016-05-06 00:00:00
12 2016-05-07 00:00:00
13 2016-05-08 00:00:00
14 2016-05-09 00:00:00
15 2

In [21]:
def rmsse(train, test, forecast):

    #Sort index để phòng bị sai thứ tự
    test = test.sort_index()
    forecast = forecast.sort_index()
    train = train.sort_index()

    #Tính toán RMSSE
    forecast_mse = np.mean((test - forecast)**2, axis=1)
    train_mse = train.apply(
        lambda row: np.mean(np.diff(np.trim_zeros(row.values)) ** 2)
        if len(np.trim_zeros(row.values)) > 1 else 0,
        axis=1
    )
    return np.sqrt(forecast_mse / train_mse)

In [22]:
#Preview dữ liệu forecast
sub.head()

,d_1914,d_1915,d_1916,d_1917,d_1918,d_1919,d_1920,d_1921,d_1922,d_1923,d_1924,d_1925,d_1926,d_1927,d_1928,d_1929,d_1930,d_1931,d_1932,d_1933,d_1934,d_1935,d_1936,d_1937,d_1938,d_1939,d_1940,d_1941
id,,,,,,,,,,,,,,,,,,,,,,,,,,,,
FOODS_1_001_CA_1,0.804077,0.792265,0.763281,0.764405,0.983937,1.083061,1.198029,1.033757,0.977757,0.905416,0.900425,1.065092,1.291474,1.145303,0.990333,0.930154,0.918564,0.931144,1.065671,1.328108,1.270246,0.927586,0.853912,0.801496,0.827293,0.966793,1.172268,1.193435
FOODS_1_001_CA_2,0.926463,1.006068,0.896102,1.038163,1.168146,1.352504,1.460816,0.946770,0.972879,0.967815,1.003869,1.190947,1.484112,1.314712,1.020276,1.010801,1.028198,1.052107,1.230996,1.559724,1.622909,1.053140,0.987467,1.011320,0.998877,1.170529,1.569784,1.440060
FOODS_1_001_CA_3,1.113022,1.065720,0.895490,0.852039,0.900217,1.158423,1.206532,1.188379,1.183703,1.009581,0.996941,1.080790,1.400160,1.212179,1.181112,1.134408,0.993458,0.979787,1.007855,1.605193,1.837181,1.151772,1.087793,0.894128,0.863079,0.936163,1.247931,1.391099
FOODS_1_001_CA_4,0.448898,0.348885,0.355739,0.360381,0.422065,0.419693,0.565709,0.409560,0.422543,0.434994,0.447065,0.434807,0.476821,0.384587,0.374756,0.395204,0.399459,0.425015,0.469094,0.510492,0.480327,0.384232,0.363705,0.358890,0.380200,0.419386,0.484549,0.466431
FOODS_1_001_TX_1,0.184206,0.181095,0.192601,0.193988,0.168686,0.160562,0.208287,0.532737,0.483866,0.515773,0.485304,0.532747,0.553098,0.444146,0.466805,0.456792,0.397093,0.403119,0.422233,0.386452,0.430684,0.287959,0.286163,0.274292,0.289301,0.314593,0.335013,0.359325


In [23]:
#Xử lý lại tập test để đánh giá
df_valid = pd.read_csv("../dataset/raw/sales_train_evaluation.csv")
cols = ["id"] + [f"d_{i}" for i in range(tr_last + 1, 1942)]
test = df_valid[cols]
test["id"] = test["id"].str.extract(r"^(.+?)_(?:validation|evaluation)$", expand=False)

test.set_index("id", inplace = True)

test.head()

,d_1914,d_1915,d_1916,d_1917,d_1918,d_1919,d_1920,d_1921,d_1922,d_1923,d_1924,d_1925,d_1926,d_1927,d_1928,d_1929,d_1930,d_1931,d_1932,d_1933,d_1934,d_1935,d_1936,d_1937,d_1938,d_1939,d_1940,d_1941
id,,,,,,,,,,,,,,,,,,,,,,,,,,,,
HOBBIES_1_001_CA_1,0,0,0,2,0,3,5,0,0,1,1,0,2,1,2,2,1,0,2,4,0,0,0,0,3,3,0,1
HOBBIES_1_002_CA_1,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,1,2,1,1,0,0,0,0,0
HOBBIES_1_003_CA_1,0,0,1,1,0,2,1,0,0,0,0,2,1,3,0,0,1,0,1,0,2,0,0,0,2,3,0,1
HOBBIES_1_004_CA_1,0,0,1,2,4,1,6,4,0,0,0,2,2,4,2,1,1,1,1,1,0,4,0,1,3,0,2,6
HOBBIES_1_005_CA_1,1,0,2,3,1,0,3,2,3,1,1,3,2,3,2,2,2,2,0,0,0,2,1,0,0,2,1,0


In [24]:
#Xử lý lại tập train để đánh giá
df_valid = pd.read_csv("../dataset/raw/sales_train_validation.csv")
df_valid["id"] = df_valid["id"].str.extract(r"^(.+?)_(?:validation|evaluation)$", expand=False)

df_valid.set_index("id", inplace = True)

# 1. Đổi đuôi id
train = df_valid.filter(like="d_")
train.head()

,d_1,d_2,d_3,d_4,d_5,d_6,d_7,d_8,d_9,d_10,d_11,d_12,d_13,d_14,d_15,d_16,d_17,d_18,d_19,d_20,d_21,d_22,d_23,d_24,d_25,...,d_1889,d_1890,d_1891,d_1892,d_1893,d_1894,d_1895,d_1896,d_1897,d_1898,d_1899,d_1900,d_1901,d_1902,d_1903,d_1904,d_1905,d_1906,d_1907,d_1908,d_1909,d_1910,d_1911,d_1912,d_1913
id,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
HOBBIES_1_001_CA_1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,1,0,4,2,3,0,1,2,0,0,0,1,1,3,0,1,1,1,3,0,1,1
HOBBIES_1_002_CA_1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0
HOBBIES_1_003_CA_1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,1,0,0,0,1,0,0,0,0,0,1,2,2,1,2,1,1,1,0,1,1,1
HOBBIES_1_004_CA_1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,...,0,3,1,2,1,3,1,0,2,5,4,2,0,3,0,1,0,5,4,1,0,1,3,7,2
HOBBIES_1_005_CA_1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,...,4,0,1,4,0,1,0,1,0,1,1,2,0,1,1,2,1,1,0,1,1,2,2,2,4


In [25]:
# Tỷ trọng
item_share = pd.read_csv("../dataset/result/item_share.csv")
item_share.set_index("item_id", inplace = True)

item_share.tail()

,sales_share
item_id,
HOBBIES_2_025_TX_1,8.218128e-08
HOUSEHOLD_1_512_CA_4,6.586242e-08
FOODS_1_079_CA_2,3.263771e-08
FOODS_1_079_CA_3,1.631885e-08
FOODS_2_394_TX_3,0.000000e+00


In [67]:
threshold = 0.55

sub_rounded = np.floor(sub) + ((sub - np.floor(sub)) >= threshold)
sub_rounded = sub_rounded.astype(int)
sub_rounded.head()

,d_1914,d_1915,d_1916,d_1917,d_1918,d_1919,d_1920,d_1921,d_1922,d_1923,d_1924,d_1925,d_1926,d_1927,d_1928,d_1929,d_1930,d_1931,d_1932,d_1933,d_1934,d_1935,d_1936,d_1937,d_1938,d_1939,d_1940,d_1941
id,,,,,,,,,,,,,,,,,,,,,,,,,,,,
FOODS_1_001_CA_1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1
FOODS_1_001_CA_2,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,2,2,1,1,1,1,1,2,1
FOODS_1_001_CA_3,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,2,2,1,1,1,1,1,1,1
FOODS_1_001_CA_4,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
FOODS_1_001_TX_1,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0


In [68]:
rmsse_result = rmsse(train, test, sub_rounded)
print(rmsse_result.mean())

0.7879453858484575


In [69]:
wrmsse = rmsse_result * item_share['sales_share']
wrmsse.sum()

np.float64(0.7628176125412578)